# Transformación de datos — Lead Scoring

Etapa de diagnóstico y diseño. **No se modifica `df` antes de congelar la matriz.**

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().parent
state_file = PROJECT_ROOT / '.github' / 'copilot-instructions.md'
state_text = state_file.read_text(encoding='utf-8')
state_section = state_text.split('## ESTADO ACTUAL DEL PROYECTO', 1)[1]
input_path_text = state_section.split('`', 2)[1]
input_path = (Path.cwd() / input_path_text).resolve()
df = pd.read_pickle(input_path)
print(f'Input path from copilot instructions: {input_path}')
print(f'Shape: {df.shape}')
df.info()

Input path from copilot instructions: C:\Users\Dell\Agus\Master Agentic DS\Lead_Scoring\02_datos\03_Entrenamiento\03_train_tablon_eda.pkl
Shape: (6360, 23)
<class 'pandas.DataFrame'>
Index: 6360 entries, 2954 to 7270
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   id                             6360 non-null   int64  
 1   origen                         6360 non-null   object 
 2   fuente                         6360 non-null   object 
 3   no_enviar_email                6360 non-null   object 
 4   no_llamar                      6360 non-null   object 
 5   compra                         6360 non-null   int64  
 6   visitas_total                  6360 non-null   float64
 7   tiempo_en_site_total           6360 non-null   int64  
 8   paginas_vistas_visita          6360 non-null   float64
 9   ult_actividad                  6360 non-null   object 
 10  ambito                   

In [2]:
# Diagnóstico automático por variable — solo lectura
def classify_column(s: pd.Series) -> str:
    name = s.name
    n_unique = s.nunique(dropna=True)
    if name == 'id' or n_unique == len(s):
        return 'id/pseudo-id'
    if pd.api.types.is_datetime64_any_dtype(s):
        return 'fecha'
    if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
        if s.dropna().astype(str).str.len().median() > 40:
            return 'texto'
        if n_unique == 2:
            return 'binaria'
        return 'cat_nominal'
    if n_unique == 2:
        return 'binaria'
    if pd.api.types.is_numeric_dtype(s):
        return 'num_discreta' if bool(np.isclose(s.dropna().to_numpy(dtype=float) % 1, 0).all()) else 'num_continua'
    return 'otro'

records = []
for col in df.columns:
    s = df[col]
    kind = classify_column(s)
    rec = {
        'variable': col,
        'tipo_propuesto': kind,
        'dtype': str(s.dtype),
        'unicos': int(s.nunique(dropna=True)),
        'missing_%': round(float(s.isna().mean() * 100), 2),
        'constante': bool(s.nunique(dropna=False) <= 1),
        'skew': round(float(s.skew()), 3) if pd.api.types.is_numeric_dtype(s) and s.nunique() > 2 else np.nan,
    }
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 2:
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        rec['outliers_iqr'] = int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())
    else:
        rec['outliers_iqr'] = np.nan
    records.append(rec)
diagnostico = pd.DataFrame(records)
display(diagnostico)
print('Categorías de trabajo:')
display(diagnostico.groupby('tipo_propuesto', dropna=False)['variable'].agg(list).rename('variables').to_frame())

,variable,tipo_propuesto,dtype,unicos,missing_%,constante,skew,outliers_iqr
0,id,id/pseudo-id,int64,6360,0.00,False,0.144,0.0
1,origen,cat_nominal,object,4,0.00,False,NaN,NaN
2,fuente,cat_nominal,object,5,0.00,False,NaN,NaN
3,no_enviar_email,binaria,object,2,0.00,False,NaN,NaN
4,no_llamar,binaria,object,2,0.00,False,NaN,NaN
5,compra,binaria,int64,2,0.00,False,NaN,NaN
6,visitas_total,num_discreta,float64,31,0.00,False,2.177,174.0
7,tiempo_en_site_total,num_discreta,int64,1570,0.00,False,0.937,0.0
8,paginas_vistas_visita,num_continua,float64,92,0.00,False,1.318,249.0
9,ult_actividad,cat_nominal,object,8,0.00,False,NaN,NaN


Categorías de trabajo:


,variables
tipo_propuesto,
binaria,"[no_enviar_email, no_llamar, compra, conociste..."
cat_nominal,"[origen, fuente, ult_actividad, ambito, ocupac..."
id/pseudo-id,[id]
num_continua,[paginas_vistas_visita]
num_discreta,"[visitas_total, tiempo_en_site_total, score_ac..."


## Ejecución posterior a la congelación

La matriz está congelada. Se ejecutan las fases `separar → transformar → unir`.

In [3]:
# FASE 0 — marcado y separación
TARGET = 'compra'
ID_COLUMN = 'id'
CONSTANT_COLUMNS = ['conociste_revista', 'conociste_periodico', 'conociste_youtube']
BEHAVIOR_COLUMNS = ['visitas_total', 'tiempo_en_site_total', 'paginas_vistas_visita']
SCORE_COLUMNS = ['score_actividad', 'score_perfil']
CATEGORICAL_COLUMNS = ['origen', 'fuente', 'no_enviar_email', 'no_llamar', 'ult_actividad', 'ambito', 'ocupacion', 'conociste_google', 'conociste_facebook', 'conociste_referencias', 'descarga_lm']
BINARY_PASSTHROUGH_COLUMNS = ['visitas_total_missing']

input_rows = len(df)
y = df[[TARGET]].copy()
X_raw = df.drop(columns=[TARGET, ID_COLUMN])
print(f'Filas: {input_rows}; target: {TARGET}')
print('Excluidas:', [ID_COLUMN, *CONSTANT_COLUMNS])
print('Numéricas a escalar:', BEHAVIOR_COLUMNS + SCORE_COLUMNS)
print('Binarias sin escalar:', BINARY_PASSTHROUGH_COLUMNS)

Filas: 6360; target: compra
Excluidas: ['id', 'conociste_revista', 'conociste_periodico', 'conociste_youtube']
Numéricas a escalar: ['visitas_total', 'tiempo_en_site_total', 'paginas_vistas_visita', 'score_actividad', 'score_perfil']
Binarias sin escalar: ['visitas_total_missing']


In [4]:
# FASE 1 — transformadores numéricos
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, MissingIndicator

behavior_pipeline = Pipeline([
    ('yeo_johnson', PowerTransformer(method='yeo-johnson', standardize=False)),
    ('minmax', MinMaxScaler()),
])
score_value_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('minmax', MinMaxScaler()),
])
print('FASE 1 lista: Yeo-Johnson + MinMax para comportamiento; mediana + MinMax para scores.')

FASE 1 lista: Yeo-Johnson + MinMax para comportamiento; mediana + MinMax para scores.


In [5]:
# FASE 2 — categóricas y binarias
preprocessor = ColumnTransformer(
    transformers=[
        ('behavior', behavior_pipeline, BEHAVIOR_COLUMNS),
        ('score_values', score_value_pipeline, SCORE_COLUMNS),
        ('score_missing', MissingIndicator(features='missing-only'), ['score_actividad']),
        ('categorical', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), CATEGORICAL_COLUMNS),
        ('binary', 'passthrough', BINARY_PASSTHROUGH_COLUMNS),
    ],
    remainder='drop',
    verbose_feature_names_out=True,
)
transformed_array = preprocessor.fit_transform(X_raw)
feature_names = list(preprocessor.get_feature_names_out())
print(f'FASE 2 lista: {len(feature_names)} features generadas; OHE con drop=first.')

FASE 2 lista: 52 features generadas; OHE con drop=first.


In [6]:
# FASE 3 — escalado selectivo
# El escalado fue encapsulado en los pipelines numéricos. Dummies, indicadores y flags quedan fuera de MinMaxScaler.
print('FASE 3 lista: solo comportamiento y valores de scores fueron escalados a [0, 1].')

FASE 3 lista: solo comportamiento y valores de scores fueron escalados a [0, 1].


In [7]:
# FASE 4 — unión, validación y persistencia
import joblib
from itertools import combinations
from pathlib import Path

name_map = {
    'behavior__visitas_total': 'visitas_total_yj_mm',
    'behavior__tiempo_en_site_total': 'tiempo_en_site_total_yj_mm',
    'behavior__paginas_vistas_visita': 'paginas_vistas_visita_yj_mm',
    'score_values__score_actividad': 'score_actividad_mm',
    'score_values__score_perfil': 'score_perfil_mm',
    'score_missing__missingindicator_score_actividad': 'score_actividad_missing',
}
final_feature_names = [name_map.get(name, name.replace('categorical__', '').replace('binary__', '')) for name in feature_names]
features = pd.DataFrame(transformed_array, index=df.index, columns=final_feature_names)
df_final = pd.concat([y, features], axis=1)

binary_columns = [c for c in features if set(features[c].dropna().unique()).issubset({0, 1})]
perfect_binary_pairs = [
    (left, right) for left, right in combinations(binary_columns, 2)
    if np.array_equal(features[left].to_numpy(), features[right].to_numpy())
    or np.array_equal(features[left].to_numpy(), 1 - features[right].to_numpy())
]
transformed_originals = set(BEHAVIOR_COLUMNS + SCORE_COLUMNS + CATEGORICAL_COLUMNS) & set(df_final.columns)
validations = {
    'filas_preservadas': len(df_final) == input_rows,
    'target_presente': TARGET in df_final.columns,
    'sin_columnas_intermedias': not any(c.endswith(('_yj', '_imputed', '_raw')) for c in df_final.columns),
    'sin_originales_transformados': not transformed_originals,
    'sin_nan_inesperados': int(df_final.isna().sum().sum()) == 0,
    'nombres_unicos': df_final.columns.is_unique,
    'sin_multicolinealidad_binaria_perfecta': len(perfect_binary_pairs) == 0,
}
assert all(validations.values()), validations

df = df_final
project_root = Path.cwd().parent
model_path = project_root / '05_modelos' / 'preprocesador.joblib'
output_path = project_root / '02_datos' / '03_Entrenamiento' / '04_train_tablon_transformado.pkl'
report_path = project_root / '06_resultados' / 'Transformacion' / 'informe_transformacion.md'
model_path.parent.mkdir(parents=True, exist_ok=True)
report_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessor, model_path)
df.to_pickle(output_path)
loaded_preprocessor = joblib.load(model_path)
roundtrip = loaded_preprocessor.transform(X_raw)
assert roundtrip.shape[1] == len(final_feature_names)

report = f'''# Informe de transformación — Lead Scoring

## Resumen

- Input: `02_datos/03_Entrenamiento/03_train_tablon_eda.pkl`
- Output: `02_datos/03_Entrenamiento/04_train_tablon_transformado.pkl`
- Filas: {df.shape[0]}
- Columnas finales: {df.shape[1]}
- Problema: clasificación binaria; target `compra`.
- Objetivo: priorizar recall, controlando precisión operativa.
- Modelo priorizado: regresión logística interpretable.

## Transformaciones

- Categóricas y binarias objeto: `OneHotEncoder(drop="first")`, sin escalado.
- Comportamiento: Yeo-Johnson y MinMaxScaler; finales `{', '.join(BEHAVIOR_COLUMNS)}` con sufijo `_yj_mm`.
- Scores: mediana, indicador estructural de ausencia y MinMaxScaler para el valor; finales `score_actividad_mm`, `score_perfil_mm`, `score_actividad_missing`.
- IDs y constantes: `id`, `conociste_revista`, `conociste_periodico` y `conociste_youtube` excluidos.

## Gestión de versiones intermedias

Los originales categóricos y numéricos transformados, junto con cualquier versión intermedia, fueron excluidos. Solo se conserva `compra`, las versiones finales y los indicadores binarios aprobados.

## Validaciones realizadas

{chr(10).join(f'- {k}: {'OK' if v else 'FALLÓ'}' for k, v in validations.items())}

## Reproducibilidad

El preprocesador ajustado exclusivamente sobre el tablón de entrenamiento se guardó en `05_modelos/preprocesador.joblib`. Su `transform(X_raw)` reproduce las {len(final_feature_names)} columnas predictoras finales.
'''
report_path.write_text(report, encoding='utf-8')

state_path = project_root / '.github' / 'copilot-instructions.md'
info_lines = []
df.info(buf=type('Buffer', (), {'write': lambda self, x: info_lines.append(x)})())
new_state = '## ESTADO ACTUAL DEL PROYECTO\n\n**Dataframe actual**: `../02_datos/03_Entrenamiento/04_train_tablon_transformado.pkl`\n\n**Estructura del dataframe**:\n```\n' + ''.join(info_lines) + '```\n'
state_path.write_text(new_state, encoding='utf-8')
print(f'FASE 4 OK: df final {df.shape}; {len(final_feature_names)} predictores; validaciones aprobadas.')
print(pd.Series(validations).to_string())

FASE 4 OK: df final (6360, 53); 52 predictores; validaciones aprobadas.
filas_preservadas                         True
target_presente                           True
sin_columnas_intermedias                  True
sin_originales_transformados              True
sin_nan_inesperados                       True
nombres_unicos                            True
sin_multicolinealidad_binaria_perfecta    True
